**PERSIAPAN**

In [ ]:
#menghubungkan notebook colab ke google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#melakukan import library
from pprint import pprint
from time import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
#menghubungkan colab agar pencarian file tertuju ke dalam my drive
path = '/content/drive/MyDrive/'
os.chdir(path)

In [ ]:
# membaca file serta mengetahui berapa lama waktu yang di butuhkan untuk menyelesaikan tahap ini
start = time()
oi = pd.read_excel("property.xlsx")
end = time()

print("Done loading all datasets in {:.3f}s".format(end-start))

Done loading all datasets in 4.267s


In [ ]:
#digunakan untuk mengetahui jumlah row dalam data
print("num of articles:", oi.shape[0])

num of articles: 5018


**CLEANING DATA**

In [ ]:
oi.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,availability_365
0,34229869,Cozy Living Room Futon,27537930,Preston,Bronx,Pelham Gardens,40.86209,-73.84667,Shared room,20.0,1,NaN,NaT,NaN,5
1,31568673,Large private room w/ 2 built-in closets,236631469,Paloma,Bronxx,Williamsbridge,40.88449,-73.86227,Private room,25.0,30,NaN,NaT,NaN,85
2,36199363,Furnished room for rent in Bronx SINGLE FEMALE,272247972,Kadeen,Bronxx,Olinville,40.88116,-73.86547,Shared room,25.0,90,NaN,NaT,NaN,190
3,34180340,Sunny & Spacious Room,181233778,Aishah,Bronxx,Belmont,40.85595,-73.88519,Private room,27.0,30,NaN,NaT,NaN,68
4,32364762,Bronx,19303369,Hiroki,Bronxx,Kingsbridge,40.88474,-73.90321,Private room,27.0,30,NaN,NaT,NaN,0


In [ ]:
oi.describe()

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,availability_365
count,5.018000e+03,5.018000e+03,5018.000000,5018.000000,4966.000000,5018.000000,4143.000000,4143,4143.000000,5018.000000
mean,2.301899e+07,1.090119e+08,40.732272,-73.853449,94.246073,5.684137,36.412020,2019-02-28 18:52:02.954381056,2.155752,165.734556
min,1.819800e+04,3.415000e+03,40.565460,-73.908270,10.000000,-1.000000,1.000000,2011-09-19 00:00:00,0.010000,0.000000
25%,1.644100e+07,2.820543e+07,40.688168,-73.894860,49.000000,1.000000,4.000000,2019-03-31 00:00:00,0.530000,36.000000
50%,2.424981e+07,9.731116e+07,40.726965,-73.867195,68.000000,2.000000,15.000000,2019-06-16 00:00:00,1.520000,150.000000
75%,3.117735e+07,1.796772e+08,40.759290,-73.820752,100.000000,3.000000,46.000000,2019-06-30 00:00:00,3.000000,312.000000
max,3.648436e+07,2.742256e+08,40.913060,-73.712990,10000.000000,5000.000000,629.000000,2019-07-08 00:00:00,20.940000,365.000000
std,9.763398e+06,8.493779e+07,0.067359,0.048489,183.560851,71.560418,56.865827,NaN,2.275825,134.476153


In [ ]:
#melihat info tabel
oi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5018 entries, 0 to 5017
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   5018 non-null   int64         
 1   name                 5017 non-null   object        
 2   host_id              5018 non-null   int64         
 3   host_name            5016 non-null   object        
 4   neighbourhood_group  5018 non-null   object        
 5   neighbourhood        5018 non-null   object        
 6   latitude             5018 non-null   float64       
 7   longitude            5018 non-null   float64       
 8   room_type            5018 non-null   object        
 9   price                4966 non-null   float64       
 10  minimum_nights       5018 non-null   int64         
 11  number_of_reviews    4143 non-null   float64       
 12  last_review          4143 non-null   datetime64[ns]
 13  reviews_per_month    4143 non-nul

In [ ]:
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                       0
name                     1
host_id                  0
host_name                2
neighbourhood_group      0
neighbourhood            0
latitude                 0
longitude                0
room_type                0
price                   52
minimum_nights           0
number_of_reviews      875
last_review            875
reviews_per_month      875
availability_365         0
dtype: int64


**DUPLICATE**

In [ ]:
#menghitung jumlah duplikat dalam DataFrame
jumlah_duplikat = oi.duplicated().sum()

#menampilkan jumlah duplikat
print("Jumlah duplikat dalam DataFrame:", jumlah_duplikat)

Jumlah duplikat dalam DataFrame: 19


In [ ]:
#menghapus baris duplikat dari DataFrame
oi_tanpa_duplikat = oi.drop_duplicates()

#menampilkan DataFrame baru tanpa duplikat
print(oi_tanpa_duplikat)

            id                                               name    host_id  \
0     34229869                             Cozy Living Room Futon   27537930   
1     31568673           Large private room w/ 2 built-in closets  236631469   
2     36199363    Furnished room  for rent in Bronx SINGLE FEMALE  272247972   
3     34180340                              Sunny & Spacious Room  181233778   
4     32364762                                              Bronx   19303369   
...        ...                                                ...        ...   
5013    546383                   My Little Guest Room in Flushing    2680820   
5014  16276632  Cozy Room Family Home LGA Airport NO CLEANING FEE   26432133   
5015   8168619                  Steps away from Laguardia airport   37312959   
5016  10101135                            Room Near JFK Twin Beds   47621202   
5017   9145202                            Room near JFK Queen Bed   47621202   

     host_name neighbourhood_group   ne

In [ ]:
#menghapus baris duplikat dari DataFrame asli
oi.drop_duplicates(inplace=True)

In [ ]:
#melakukan cek jumlah duplikat
jumlah_duplikat = oi.duplicated().sum()

#menampilkan jumlah duplikat
print("Jumlah duplikat dalam DataFrame:", jumlah_duplikat)

Jumlah duplikat dalam DataFrame: 0


In [ ]:
#melihat info tabel
oi.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4999 entries, 0 to 5017
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   4999 non-null   int64         
 1   name                 4998 non-null   object        
 2   host_id              4999 non-null   int64         
 3   host_name            4997 non-null   object        
 4   neighbourhood_group  4999 non-null   object        
 5   neighbourhood        4999 non-null   object        
 6   latitude             4999 non-null   float64       
 7   longitude            4999 non-null   float64       
 8   room_type            4999 non-null   object        
 9   price                4947 non-null   float64       
 10  minimum_nights       4999 non-null   int64         
 11  number_of_reviews    4125 non-null   float64       
 12  last_review          4125 non-null   datetime64[ns]
 13  reviews_per_month    4125 non-null   f

**OUTLIERS**

**price**

In [ ]:
#melihat jumlah outliers
Q1 = oi['price'].quantile(0.25)
Q3 = oi['price'].quantile(0.75)
IQR = Q3 - Q1
#pendefinisian
b_atas = Q3+(1.5*IQR)
b_bawah = Q1-(1.5*IQR)
#melakukan penghapusan outlier
kurang_dari = oi['price'] < b_bawah
lebih_dari = oi['price'] > b_atas

oi = oi.loc[(oi['price'] >= b_bawah) & (oi['price'] <= b_atas)]

In [ ]:
#melihat info outliers
oi.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4546 entries, 0 to 5017
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   4546 non-null   int64         
 1   name                 4545 non-null   object        
 2   host_id              4546 non-null   int64         
 3   host_name            4544 non-null   object        
 4   neighbourhood_group  4546 non-null   object        
 5   neighbourhood        4546 non-null   object        
 6   latitude             4546 non-null   float64       
 7   longitude            4546 non-null   float64       
 8   room_type            4546 non-null   object        
 9   price                4546 non-null   float64       
 10  minimum_nights       4546 non-null   int64         
 11  number_of_reviews    3783 non-null   float64       
 12  last_review          3783 non-null   datetime64[ns]
 13  reviews_per_month    3783 non-null   f

**minimum_nights**

In [ ]:
#melihat jumlah outliers
Q1 = oi['minimum_nights'].quantile(0.25)
Q3 = oi['minimum_nights'].quantile(0.75)
IQR = Q3 - Q1
#pendefinisian
b_atas = Q3+(1.5*IQR)
b_bawah = Q1-(1.5*IQR)
#melakukan penghapusan outliers
kurang_dari = oi['minimum_nights'] < b_bawah
lebih_dari = oi['minimum_nights'] > b_atas

oi = oi.loc[(oi['minimum_nights'] >= b_bawah) & (oi['minimum_nights'] <= b_atas)]

In [ ]:
#melihat info outliers
oi.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3970 entries, 0 to 5017
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   3970 non-null   int64         
 1   name                 3969 non-null   object        
 2   host_id              3970 non-null   int64         
 3   host_name            3968 non-null   object        
 4   neighbourhood_group  3970 non-null   object        
 5   neighbourhood        3970 non-null   object        
 6   latitude             3970 non-null   float64       
 7   longitude            3970 non-null   float64       
 8   room_type            3970 non-null   object        
 9   price                3970 non-null   float64       
 10  minimum_nights       3970 non-null   int64         
 11  number_of_reviews    3424 non-null   float64       
 12  last_review          3424 non-null   datetime64[ns]
 13  reviews_per_month    3424 non-null   f

**MISSING VALUE**

In [ ]:
#melihat value yang missing
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                       0
name                     1
host_id                  0
host_name                2
neighbourhood_group      0
neighbourhood            0
latitude                 0
longitude                0
room_type                0
price                    0
minimum_nights           0
number_of_reviews      546
last_review            546
reviews_per_month      546
availability_365         0
dtype: int64


**tabel name**

In [ ]:
#menganti missing value di tabel name dengan Hillside Hotel karna modus
oi['name'].fillna('Hillside Hotel', inplace=True)

In [ ]:
#melakukan cek name
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                       0
name                     0
host_id                  0
host_name                2
neighbourhood_group      0
neighbourhood            0
latitude                 0
longitude                0
room_type                0
price                    0
minimum_nights           0
number_of_reviews      546
last_review            546
reviews_per_month      546
availability_365         0
dtype: int64


**tabel host_name**

In [ ]:
#menganti missing value di tabel host_name dengan Kazuya karna modus
oi['host_name'].fillna('Kazuya', inplace=True)

In [ ]:
#melakukan cek host_name
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                       0
name                     0
host_id                  0
host_name                0
neighbourhood_group      0
neighbourhood            0
latitude                 0
longitude                0
room_type                0
price                    0
minimum_nights           0
number_of_reviews      546
last_review            546
reviews_per_month      546
availability_365         0
dtype: int64


**tabel price**

In [ ]:
#mengetahui median price
median = oi['price'].median()
print("Nilai median dari kolom 'price':", median)

Nilai median dari kolom 'price': 65.0


In [ ]:
#menganti missing value di tabel price dengan 68 karna median
oi['price'].fillna('68', inplace=True)

In [ ]:
#melakukan cek price
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                       0
name                     0
host_id                  0
host_name                0
neighbourhood_group      0
neighbourhood            0
latitude                 0
longitude                0
room_type                0
price                    0
minimum_nights           0
number_of_reviews      546
last_review            546
reviews_per_month      546
availability_365         0
dtype: int64


**number_of_revies**

In [ ]:
#mengetahui median number_of_reviews
median = oi['number_of_reviews'].median()
print("Nilai median dari kolom 'number_of_reviews':", median)

Nilai median dari kolom 'number_of_reviews': 18.0


In [ ]:
#menganti missing value di tabel number_of_reviews dengan 15 karna median
oi['number_of_reviews'].fillna('15', inplace=True)

In [ ]:
#melakukan cek number_of_reviews
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                       0
name                     0
host_id                  0
host_name                0
neighbourhood_group      0
neighbourhood            0
latitude                 0
longitude                0
room_type                0
price                    0
minimum_nights           0
number_of_reviews        0
last_review            546
reviews_per_month      546
availability_365         0
dtype: int64


**last_review**

In [ ]:
#mengetahui median last_review
median = oi['last_review'].median()
print("Nilai median dari kolom 'last_review':", median)

Nilai median dari kolom 'last_review': 2019-06-17 00:00:00


In [ ]:
#menganti missing value di tabel last_review dengan 2019-06-16 karna median
oi['last_review'].fillna('2019-06-16', inplace=True)

In [ ]:
#melakukan cek last_review
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                       0
name                     0
host_id                  0
host_name                0
neighbourhood_group      0
neighbourhood            0
latitude                 0
longitude                0
room_type                0
price                    0
minimum_nights           0
number_of_reviews        0
last_review              0
reviews_per_month      546
availability_365         0
dtype: int64


**reviews_per_month**

In [ ]:
#mengetahui median reviews_per_month
median = oi['reviews_per_month'].median()
print("Nilai median dari kolom 'reviews_per_month':", median)

Nilai median dari kolom 'reviews_per_month': 1.73


In [ ]:
#menganti missing value di tabel reviews_per_month dengan 2019-06-16 karna median
oi['reviews_per_month'].fillna('1.52', inplace=True)

In [ ]:
#melakukan cek reviews_per_month
missing_values_per_column = oi.isnull().sum()

#menampilkan jumlah baris dengan nilai kosong di setiap kolom
print("jumlah nilai kosong di setiap kolom:")
print(missing_values_per_column)

jumlah nilai kosong di setiap kolom:
id                     0
name                   0
host_id                0
host_name              0
neighbourhood_group    0
neighbourhood          0
latitude               0
longitude              0
room_type              0
price                  0
minimum_nights         0
number_of_reviews      0
last_review            0
reviews_per_month      0
availability_365       0
dtype: int64


**TYPOS**

In [ ]:
#melihat isi tabel neighbourhood_group sebelum dirubah
for col in ['neighbourhood_group']:
  print(oi[col].unique())

['Bronx' 'Bronxx' 'Brooklyn' 'Brooklynn' 'Queens' 'QUeens']


In [ ]:
oi.replace({'Bronxx': 'Bronx', 'Brooklynn': 'Brooklyn', 'QUeens': 'Queens'}, inplace=True)

In [ ]:
#melihat isi tabel neighbourhood_group yang telah di rubah
for col in ['neighbourhood_group']:
  print(oi[col].unique())

['Bronx' 'Brooklyn' 'Queens']


**melihat info tabel setelah dicleaning**

In [ ]:
#melihat info tabel
oi.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3970 entries, 0 to 5017
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   3970 non-null   int64         
 1   name                 3970 non-null   object        
 2   host_id              3970 non-null   int64         
 3   host_name            3970 non-null   object        
 4   neighbourhood_group  3970 non-null   object        
 5   neighbourhood        3970 non-null   object        
 6   latitude             3970 non-null   float64       
 7   longitude            3970 non-null   float64       
 8   room_type            3970 non-null   object        
 9   price                3970 non-null   float64       
 10  minimum_nights       3970 non-null   int64         
 11  number_of_reviews    3970 non-null   object        
 12  last_review          3970 non-null   datetime64[ns]
 13  reviews_per_month    3970 non-null   o

**1. Harga rata-rata, termurah dan termahal yang disediakan per jenis tipe kamar beserta karakteristiknya**

In [ ]:
# Mengonversi kolom harga dan jumlah ulasan ke tipe data numerik
oi['price'] = oi['price'].replace('[\$,]', '', regex=True).astype(float)
oi['number_of_reviews'] = oi['number_of_reviews'].astype(float)

# Mendapatkan harga rata-rata, maksimum, dan minimum per jenis tipe kamar
room_prices = oi.groupby('room_type').agg({'price': ['mean', 'max', 'min'],
                                           'minimum_nights': 'mean',
                                           'number_of_reviews': 'mean'})

# Menampilkan hasil
print(room_prices)

                      price              minimum_nights number_of_reviews
                       mean    max   min           mean              mean
room_type                                                                
Entire home/apt  103.961631  175.0  10.0       2.183054         44.501998
Private room      62.037383  175.0  21.0       1.802960         33.290888
Shared room       46.682119  159.0  11.0       1.582781         18.900662


**2. Total properti dan rata-rata harga per neighbourhood group**

In [ ]:
#group berdasarkan neighbourhood_group dan hitung total properti dan rata-rata harga
result_2 = oi.groupby('neighbourhood_group').agg({'id': 'count', 'price': 'mean'})

#rename columns
result_2.columns = ['total_properti', 'avg_price_properti']

#reset index
result_2 = result_2.reset_index()

#print result
print(result_2)

  neighbourhood_group  total_properti  avg_price_properti
0               Bronx             643           70.270607
1            Brooklyn             602           78.081395
2              Queens            2725           74.946055


**3. Dari tahun 2017-2019, berapa rata-rata reviews per month dan rata-rata availability kamar nya untuk setiap jenis tipe kamar per tahunnya**

In [ ]:
#mengubah last_review ke format datetime
oi['last_review'] = pd.to_datetime(oi['last_review'])

#mengambil data dari tahun 2017-2019
data_filtered = oi[oi['last_review'].dt.year.between(2017, 2019)]

#group berdasarkan year, room_type, dan hitung rata-rata reviews_per_month dan availability_365
result_3 = data_filtered.groupby([data_filtered['last_review'].dt.year, 'room_type']).agg({'number_of_reviews': 'mean', 'availability_365': 'mean'})

#reset index
result_3 = result_3.reset_index()

#rename columns
result_3.columns = ['year', 'room_type', 'avg_review_per_month', 'avg_availability']

#print result
print(result_3)

   year        room_type  avg_review_per_month  avg_availability
0  2017  Entire home/apt             11.647059         40.529412
1  2017     Private room             12.313725         69.627451
2  2017      Shared room              5.500000         44.500000
3  2018  Entire home/apt             14.026667         38.186667
4  2018     Private room             18.022727        122.920455
5  2018      Shared room              6.200000        152.266667
6  2019  Entire home/apt             48.453894        167.019696
7  2019     Private room             37.082044        174.121894
8  2019      Shared room             20.877863        210.671756
